# Grok-rl-02-mdp-dp

**Stage 02 — MDP & Dynamic Programming**

## 概念
Bandit 没有“状态”。真实决策是：**状态 s → 动作 a → 下一状态 s' + 奖励 r**。  
若已知转移与奖励模型，可用 **Bellman 最优方程** 精确求解。

## 算法
1. Policy Evaluation
2. Policy Iteration
3. Value Iteration

## 环境
自定义 GridWorld（悬崖/障碍/目标）— from scratch。


In [ ]:

import json, time, platform
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
SEED = 0
rng = np.random.default_rng(SEED)

gpu_info = {"cuda": False, "device_count": 0, "names": []}
try:
    import torch
    gpu_info["cuda"] = torch.cuda.is_available()
    gpu_info["device_count"] = torch.cuda.device_count() if gpu_info["cuda"] else 0
    gpu_info["names"] = [torch.cuda.get_device_name(i) for i in range(gpu_info["device_count"])]
except Exception as e:
    gpu_info["error"] = str(e)
print("GPU", gpu_info)


In [ ]:

# ---- GridWorld MDP (from scratch) ----
# Actions: 0U 1R 2D 3L
ACTS = {0:(-1,0), 1:(0,1), 2:(1,0), 3:(0,-1)}
ACT_NAMES = ["U","R","D","L"]

class GridWorld:
    def __init__(self, H=4, W=12, slip=0.1):
        self.H, self.W, self.slip = H, W, slip
        self.start = (3, 0)
        self.goal = (3, 11)
        self.cliff = {(3, c) for c in range(1, 11)}
        self.nS = H * W
        self.nA = 4
        # Precompute P[s][a] = list of (prob, s', r, done)
        self.P = {s: {a: [] for a in range(4)} for s in range(self.nS)}
        self._build()

    def rc(self, s):
        return divmod(s, self.W)

    def s_id(self, r, c):
        return r * self.W + c

    def _build(self):
        for s in range(self.nS):
            r, c = self.rc(s)
            terminal = (r, c) == self.goal or (r, c) in self.cliff
            for a in range(4):
                if terminal:
                    self.P[s][a] = [(1.0, s, 0.0, True)]
                    continue
                # intended + slips to perpendicular
                intents = [a]
                # slip: with prob slip, random among 4
                outcomes = {}
                for ap, p in [(a, 1 - self.slip)] + [(aa, self.slip / 4) for aa in range(4)]:
                    # Actually mix: (1-slip)*intended + slip*uniform
                    pass
                # clearer model
                dist = np.full(4, self.slip / 4.0)
                dist[a] += 1 - self.slip
                for ap, p in enumerate(dist):
                    if p <= 0: continue
                    dr, dc = ACTS[ap]
                    nr, nc = r + dr, c + dc
                    # walls: stay
                    if not (0 <= nr < self.H and 0 <= nc < self.W):
                        nr, nc = r, c
                    ns = self.s_id(nr, nc)
                    done = False
                    if (nr, nc) == self.goal:
                        rew, done = 10.0, True
                    elif (nr, nc) in self.cliff:
                        rew, done = -100.0, True
                    else:
                        rew = -1.0
                    outcomes[ns, rew, done] = outcomes.get((ns, rew, done), 0.0) + p
                self.P[s][a] = [(p, ns, rew, done) for (ns, rew, done), p in outcomes.items()]

    def render_values(self, V, policy=None, title=""):
        grid = V.reshape(self.H, self.W)
        fig, ax = plt.subplots(figsize=(10, 3))
        im = ax.imshow(grid, cmap="RdYlGn")
        for r in range(self.H):
            for c in range(self.W):
                s = self.s_id(r, c)
                txt = f"{V[s]:.1f}"
                if policy is not None and (r,c) != self.goal and (r,c) not in self.cliff:
                    txt += f"\n{ACT_NAMES[policy[s]]}"
                if (r,c) in self.cliff:
                    txt = "CLIFF"
                if (r,c) == self.goal:
                    txt = "GOAL"
                if (r,c) == self.start:
                    txt = "S\n" + txt
                ax.text(c, r, txt, ha="center", va="center", fontsize=7, color="k")
        ax.set_title(title)
        fig.colorbar(im, ax=ax, fraction=0.03)
        fig.tight_layout()
        return fig

env = GridWorld(slip=0.1)
print("nS", env.nS, "nA", env.nA, "start", env.start, "goal", env.goal)


In [ ]:

GAMMA = 0.99
THETA = 1e-6

def policy_evaluation(env, pi, gamma=GAMMA, theta=THETA):
    V = np.zeros(env.nS)
    sweeps = 0
    while True:
        delta = 0.0
        for s in range(env.nS):
            a = pi[s]
            v = 0.0
            for p, ns, r, done in env.P[s][a]:
                v += p * (r + (0.0 if done else gamma * V[ns]))
            delta = max(delta, abs(v - V[s]))
            V[s] = v
        sweeps += 1
        if delta < theta:
            break
    return V, sweeps

def policy_improvement(env, V, gamma=GAMMA):
    pi = np.zeros(env.nS, dtype=int)
    for s in range(env.nS):
        q = np.zeros(env.nA)
        for a in range(env.nA):
            for p, ns, r, done in env.P[s][a]:
                q[a] += p * (r + (0.0 if done else gamma * V[ns]))
        pi[s] = int(np.argmax(q))
    return pi

def policy_iteration(env, gamma=GAMMA):
    pi = np.zeros(env.nS, dtype=int)  # always Up initially — bad near cliff
    hist = []
    for it in range(100):
        V, sweeps = policy_evaluation(env, pi, gamma)
        new_pi = policy_improvement(env, V, gamma)
        hist.append({"iter": it, "sweeps": sweeps, "V_start": float(V[env.s_id(*env.start)])})
        if np.array_equal(new_pi, pi):
            return V, pi, hist
        pi = new_pi
    return V, pi, hist

def value_iteration(env, gamma=GAMMA, theta=THETA):
    V = np.zeros(env.nS)
    hist = []
    for it in range(10_000):
        delta = 0.0
        for s in range(env.nS):
            q = np.zeros(env.nA)
            for a in range(env.nA):
                for p, ns, r, done in env.P[s][a]:
                    q[a] += p * (r + (0.0 if done else gamma * V[ns]))
            v = q.max()
            delta = max(delta, abs(v - V[s]))
            V[s] = v
        hist.append({"iter": it, "delta": delta, "V_start": float(V[env.s_id(*env.start)])})
        if delta < theta:
            break
    pi = policy_improvement(env, V, gamma)
    return V, pi, hist

t0 = time.time()
V_pi, pi_pi, hist_pi = policy_iteration(env)
V_vi, pi_vi, hist_vi = value_iteration(env)
elapsed = time.time() - t0
print("PI iters", len(hist_pi), "V(s0)", hist_pi[-1]["V_start"])
print("VI iters", len(hist_vi), "V(s0)", hist_vi[-1]["V_start"])
print("policies equal?", np.array_equal(pi_pi, pi_vi))
print("elapsed", elapsed)


In [ ]:

# Compare to random policy value
pi_rand = rng.integers(0, 4, size=env.nS)
V_rand, sweeps_rand = policy_evaluation(env, pi_rand)
print("random V(s0)", V_rand[env.s_id(*env.start)], "sweeps", sweeps_rand)
print("optimal V(s0)", V_vi[env.s_id(*env.start)])

fig1 = env.render_values(V_vi, pi_vi, "Value Iteration — V* + greedy policy")
fig1.savefig(OUT/"stage02_vi_policy.png", dpi=120); plt.close(fig1)
fig2 = env.render_values(V_rand, pi_rand, "Random policy evaluation V^π")
fig2.savefig(OUT/"stage02_random_policy.png", dpi=120); plt.close(fig2)

# convergence curve
fig, ax = plt.subplots(figsize=(6,3))
ax.plot([h["V_start"] for h in hist_vi], label="VI V(start)")
ax.plot([h["V_start"] for h in hist_pi], label="PI V(start)")
ax.axhline(V_rand[env.s_id(*env.start)], ls="--", c="gray", label="random π")
ax.set_xlabel("iteration"); ax.set_ylabel("V(start)"); ax.legend(); ax.set_title("DP convergence")
fig.tight_layout(); fig.savefig(OUT/"stage02_convergence.png", dpi=120); plt.close(fig)

payload = {
  "ok": True,
  "stage": "02-mdp-dp",
  "title": "Grok-rl-02-mdp-dp",
  "gamma": GAMMA,
  "slip": env.slip,
  "V_start_optimal": float(V_vi[env.s_id(*env.start)]),
  "V_start_random": float(V_rand[env.s_id(*env.start)]),
  "pi_vi_path_actions": [ACT_NAMES[pi_vi[env.s_id(3,c)]] if (3,c) not in env.cliff and (3,c)!=env.goal else "X" for c in range(env.W)],
  "policy_iteration_iters": len(hist_pi),
  "value_iteration_iters": len(hist_vi),
  "policies_match": bool(np.array_equal(pi_pi, pi_vi)),
  "gpu": gpu_info,
  "elapsed_sec": elapsed,
  "concept": "with known model, Bellman optimality yields exact V* and pi*",
  "new_capability": "plan optimal path around cliff without trial-and-error interaction",
  "compare_to_previous": "Stage01 chose single best arm; Stage02 sequences actions over states with long-horizon value",
}
assert payload["V_start_optimal"] > payload["V_start_random"] + 5
(OUT/"results_stage02.json").write_text(json.dumps(payload, indent=2))
print(json.dumps(payload, indent=2)[:1500])
print("STAGE02_OK")
